# GistKnit AI - Model Server (Colab)
Ye notebook tumhara apna open model (Qwen2.5 7B) Colab par chalata hai aur ngrok se uska address deta hai.

**Pehle:** menu me *Runtime > Change runtime type > T4 GPU* chuno, phir Save.
**Phir:** neeche ke cells ek ek karke (ya *Runtime > Run all*) chalao. Tab band mat karna.

In [ ]:
# 1) Ollama install (2-3 minute lagte hain)
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install pyngrok

In [ ]:
# 2) Server start + model download (pehli baar 5-10 minute)
import subprocess, os, time
env = dict(os.environ, OLLAMA_KEEP_ALIVE="24h")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)
!ollama pull qwen2.5:7b

In [ ]:
# 3) Public address (ngrok)
from getpass import getpass
from pyngrok import ngrok
ngrok.kill()
ngrok.set_auth_token(getpass("ngrok Authtoken yahan paste karo, phir Enter: "))
tunnel = ngrok.connect(11434, host_header="rewrite")
url = tunnel.public_url
print("\nRender me ye 2 values daalo:")
print("LLM_BASE_URL =", url + "/v1")
print("LLM_MODEL    = qwen2.5:7b")

In [ ]:
# 4) Test: ek chhota sawal
import json, urllib.request
req = urllib.request.Request(
    url + "/v1/chat/completions",
    data=json.dumps({"model": "qwen2.5:7b", "messages": [{"role": "user", "content": "Namaste, ek line me apna parichay do."}]}).encode(),
    headers={"Content-Type": "application/json", "ngrok-skip-browser-warning": "true"},
)
print(json.load(urllib.request.urlopen(req, timeout=180))["choices"][0]["message"]["content"])

In [ ]:
# 5) Server chalta rakho. Is cell ko chalta chhod do aur tab khula rakho.
import time
while True:
    time.sleep(60)
    print("running...", time.strftime("%H:%M"))